# Lecture 01: Introduction and Tokenization

CS40008.01 · Baojian Zhou · Fudan University · September 9, 2026

**Question:** How does an LLM application turn our text into something a model can predict?

The four sections follow the slides. Predict each result before running its check.
E01–E06 and the two short practices P01–P02 are **ungraded classroom work**.

| Section | Notebook work |
| --- | --- |
| [1. Course Overview](#course-overview) | Course map, local Qwen call, E01 sentiment, E02 translation, application gallery |
| [2. Development of NLP & LLMs](#development) | Ten historical landmarks; 2026 models, scientific evaluation, and research |
| [3. Basics for Text Preprocessing](#preprocessing) | Regex P01, preserving text, Unicode E03 |
| [4. Text Tokenization](#tokenization) | Tokens/types P02; BPE training E04, encoding E05, comparison E06 |

Use the personal copy under `workspace/`. The slide toolbar's **Notebook** link creates
or reopens it in JupyterLab. To receive an updated handout, rename an existing personal
notebook, then click **Notebook** again. Keep the renamed copy for earlier answers.

The core text-processing and BPE exercises run offline. Ollama calls need a separately
installed local model. Vision, diffusion, thinking, and extra generation are optional.
No notebook cell downloads a model automatically.

<a id="course-overview"></a>
## 1. Course Overview

**NLP** develops computational methods for analyzing and generating human language.
**LLMs** are large neural language models that learn from text and can support many NLP tasks.
An application combines a model with an interface and, sometimes, retrieval or tools.

In this course, we understand mechanisms, implement small systems, and evaluate them
through controlled experiments. For each idea: **read → implement → experiment → explain**.

| Progression | Main topics |
| --- | --- |
| Represent text | Text preprocessing, tokenization, embeddings |
| Build language models | Probability, n-grams, neural models, RNN motivation, attention, Transformers |
| Train and evaluate | Data quality, optimization, compute, fine-tuning, benchmarks |
| Apply and investigate | Prompting, alignment, retrieval, efficient inference, diffusion LMs, reasoning, agents |

The [Fall course website](https://baojian.github.io/llm-26-fall/) is authoritative for
schedule and assessment. Quizzes contribute **10%**, assignments **45%**, and the
**individual** course project **45%**. A1 is released in **Week 2**. Today's exercises
are practice, not graded assignments. Explain your own experimental choices and verify
any AI-assisted result, following the course policy.

**Instructor:** Baojian Zhou · bjzhou@fudan.edu.cn · Francis and Rose Yuen Campus, C611.
Office hours: **Monday 14:00–15:30**, as listed on the Fall course page.

Python and introductory ML, including probability and matrix operations, are prerequisites.
When you get stuck, record a small example, the expected behavior, and the actual result.

### Course materials and optional resources

From the repository root, update and start the local preview:

```sh
git pull
uv sync
uv run python scripts/slides.py serve
```

Open the local course page printed by the server, then use the Lecture 01 slide and
notebook links. Keep experiments and solutions in `workspace/`.

The slide deck presents three starting points. Here is the full companion inventory
adapted from the Spring lecture; these are **optional**, not extra required courses.

- **Books:** [Speech and Language Processing](https://web.stanford.edu/~jurafsky/slp3/)
  (Jurafsky and Martin); [Foundations of Large Language Models](https://arxiv.org/abs/2501.09223)
  (Xiao and Zhu); [Introduction to NLP](https://mitpress.mit.edu/9780262042840/introduction-to-natural-language-processing/)
  (Eisenstein); [Introduction to NLP](https://intro-nlp.github.io/) (Zhang, Gui, and Huang).
- **Courses:** [Stanford CS336](https://cs336.stanford.edu/),
  [CS224N](https://web.stanford.edu/class/cs224n/), [CMU 11-711](https://cmu-l3.github.io/anlp-fall2025/),
  [UMass CS685](https://people.cs.umass.edu/~miyyer/cs685/),
  [Princeton COS 484](https://princeton-nlp.github.io/cos484/),
  [Stanford CS124](https://web.stanford.edu/class/cs124/),
  [Neural Networks: Zero to Hero](https://karpathy.ai/zero-to-hero.html).
- **Research communities:** NLP: ACL, EMNLP, NAACL, EACL, COLING
  ([ACL Anthology](https://aclanthology.org/)); ML: NeurIPS, ICML, ICLR;
  IR: SIGIR, WWW, WSDM, CIKM; data mining: KDD.

Choose a resource to answer a specific question. Begin with the assigned reading.

### The apps you use

Choose at most two apps in the [first-lecture survey](https://github.com/baojian/llm-26-fall/issues/6)
and follow its pull-request instructions after class. **Your observation:** what do you
ask the app to do, and which part might be the model, a tool, or the interface?

### App, model, tokenizer

Our interface is **Jupyter**; the runtime is **Ollama**; the installed model is a **Qwen3**
variant; its paired tokenizer defines the token IDs that its embedding table expects.
The byte BPE tokenizer we build later is a separate teaching model.

### Before running the local-model examples

The course's `uv` environment supplies Python and JupyterLab. **Ollama is a separate
application**, not a Python dependency. Install it from [ollama.com](https://ollama.com/download),
start the app (or run `ollama serve` in another terminal), then run:

```sh
ollama pull qwen3:0.6b
ollama list
```

Download the model before class; cells below never install or download an Ollama model.
The small model makes classroom experiments manageable, but it can make mistakes.
Use `qwen3:1.7b` instead if it is already installed and your computer can run it.

The notebook sends requests to your local Ollama server at `127.0.0.1:11434`. It needs no
API key or extra Python SDK. If Ollama is unavailable, the model demonstrations explain
why they are skipped; **all core tokenization exercises still run offline**.

In [ ]:
import base64
import json
import math
import os
import socket
import unicodedata
from collections import Counter
from pathlib import Path
from urllib.error import HTTPError, URLError
from urllib.request import ProxyHandler, Request, build_opener

In [ ]:
OLLAMA_URL = "http://127.0.0.1:11434"
MODEL = "qwen3:0.6b"
RUN_OLLAMA = os.environ.get("COURSE_RUN_OLLAMA", "1") == "1"
local_http = build_opener(ProxyHandler({}))  # Do not proxy local requests.


def ollama_request(endpoint, payload=None, timeout=120):
    """GET model information or POST one non-streaming JSON request."""
    data = None if payload is None else json.dumps(payload).encode("utf-8")
    request = Request(OLLAMA_URL + endpoint, data=data,
                      headers={"Content-Type": "application/json"})
    try:
        with local_http.open(request, timeout=timeout) as response:
            result = json.load(response)
    except HTTPError as error:
        detail = error.read().decode("utf-8", errors="replace")
        raise RuntimeError(f"Ollama HTTP {error.code}: {detail}") from error
    except (URLError, socket.timeout, TimeoutError, ConnectionError) as error:
        raise RuntimeError("Cannot reach Ollama, or the request timed out. "
                           "Start Ollama and check the installed model.") from error
    if not isinstance(result, dict):
        raise RuntimeError("Expected a JSON object from Ollama.")
    if result.get("error"):
        raise RuntimeError(f"Ollama: {result['error']}")
    return result

In [ ]:
OLLAMA_READY = False
if RUN_OLLAMA:
    try:
        installed_models = ollama_request("/api/tags", timeout=3)["models"]
        selected = next((m for m in installed_models if m["name"] == MODEL), None)
        if selected:
            OLLAMA_READY = True
            print("Model:", MODEL)
            print("Digest:", selected["digest"])
        else:
            print(f"Install the selected model in a terminal: ollama pull {MODEL}")
            print("Already installed:", [m["name"] for m in installed_models])
    except RuntimeError as error:
        print(error)
else:
    print("Ollama calls disabled for this run.")
if not OLLAMA_READY:
    print("Continue with predictions and E03–E06; they do not require Ollama.")

### One prompt, one response

The request sets `stream=False` to receive one JSON object. `think=False` disables the
Qwen3 thinking output for these short classroom calls. `temperature=0` selects greedy
generation; this is useful for comparisons but does not guarantee identical output across
different model files, runtimes, or hardware. `num_predict` limits generated tokens.

First predict the answer. Then inspect the **actual** response, its token counts, and its
stop reason. `prompt_eval_count` includes the model's prompt formatting; it is not a
standalone tokenization of just the visible prompt.

In [ ]:
first_response = None
if OLLAMA_READY:
    first_response = ollama_request("/api/generate", {
        "model": MODEL,
        "prompt": "Fudan University is located in which city? Answer with one word.",
        "stream": False,
        "think": False,
        "options": {"temperature": 0, "seed": 42, "num_predict": 48},
    })
    print(first_response["response"])
    print({key: first_response.get(key) for key in
           ["prompt_eval_count", "eval_count", "done_reason"]})

**Check:** the expected city is **Shanghai**. If the model names another city,
record that factual error. A confident or short answer can still be wrong.
The optional log-probability experiment revisits this exact prompt.

In [ ]:
def generate(prompt, *, temperature=0, max_tokens=96, **extra):
    """Reuse the same model and bounded settings for the next experiments."""
    if not OLLAMA_READY:
        print("Skipped model call: Ollama/model is unavailable or calls are disabled.")
        return None
    payload = {
        "model": MODEL, "prompt": prompt, "stream": False, "think": False,
        "options": {"temperature": temperature, "seed": 42,
                    "num_predict": max_tokens},
    }
    payload.update(extra)
    result = ollama_request("/api/generate", payload)
    print(result.get("response", ""))
    if result.get("done_reason") == "length":
        print("[Token limit reached: the answer may be incomplete.]")
    return result

### Language and multimodal examples

The gallery follows the same five task groups as the slides:

1. **Sentiment**: camera reviews → labels and explanations (E01).
2. **Translation**: Chinese passages → English, checking numbers and meaning (E02).
3. **Article generation**: guess authorship of two source examples; optionally generate text.
4. **Image understanding**: inspect a Big Data illustration and distinguish evidence from inference.
5. **Text to image / video**: an optional local diffusion experiment and the bundled Johannesburg clip.

For every output, ask what evidence would establish success. A plausible answer, valid
JSON, or convincing picture alone is not a complete evaluation.

### E01 · Sentiment and context · 5 minutes

These camera reviews come from the previous Fudan Lecture 01. Label each as **Positive**,
**Negative**, or **Neutral** before running the model. Explain how “light” changes meaning.

1. Nice and compact to carry!
2. Since the camera is small and light, I won't need to carry around those heavy, bulky professional cameras either!
3. The camera feels flimsy, is plastic and very light in weight you have to be very delicate in the handling of this camera.

**Your predictions and reasons:** replace this sentence with your notes.

In [ ]:
reviews = [
    "Nice and compact to carry!",
    "Since the camera is small and light, I won't need to carry around those "
    "heavy, bulky professional cameras either!",
    "The camera feels flimsy, is plastic and very light in weight you have to "
    "be very delicate in the handling of this camera.",
]
sentiment_results = []
for review in reviews:
    print("\nReview:", review)
    result = generate(
        "Classify the sentiment as Positive, Negative, or Neutral. "
        "Return JSON with keys label and reason. Use at most 12 words for the reason. "
        "Text: " + review, format="json",
    )
    sentiment_results.append(result)
    if result:
        try:
            answer = json.loads(result["response"])
            valid = (isinstance(answer, dict)
                     and answer.get("label") in {"Positive", "Negative", "Neutral"}
                     and isinstance(answer.get("reason"), str))
            print("Expected label/reason structure:", valid)
            if valid:
                print("Reason uses at most 12 words:", len(answer["reason"].split()) <= 12)
        except (json.JSONDecodeError, KeyError):
            print("The response was incomplete or not valid JSON; inspect it above.")

**Check:** expected human labels are Positive, Positive, Negative. “Light” can describe
portability or flimsy construction. Model output can disagree. Valid JSON checks the
response format, not its correctness. Three examples are enough to discuss behavior,
but not to estimate real-world accuracy.

Change only the prompt, then rerun. Did the labels change? Keep your original observations
so you can compare. Do not treat a model's self-reported confidence as calibrated accuracy.

### E02 · Translate numbers and meaning · 3 minutes

Translate into English. Use **A as the three-minute core exercise** and **B as a
follow-on comparison if time permits**. The next cell already translates both complete
passages; use those responses without making additional model calls. These A/B examples
match the combined Task 2 slide and come from
[the original translation slide](https://baojian.github.io/llm-26/slides/lecture-01-slides/index.html#/7).

### A: Numbers and dates

> Google Translate支持249种语言。日均用户超过2亿人，2016年4月总用户数超过5亿人，每天翻译超过1000亿个单词。

Before running the next cell, predict the four English quantities and the date. Then
compare with the first model response. Check language count, daily users, date, total
users, and words translated per day. Preserve what each number measures and whether
the source says “more than.” These are historical translation examples, not current
or independently verified statistics.

**A — your English quantities and date:** replace this sentence before running the cell.

**A — your comparison with the model:** record one meaning it preserved or one error you found.

### B: Definition and prediction

> 人工智能亦称智械、机器智能，指由人制造出来的机器所表现出来的智能。通常人工智能是指通过普通计算机程序来呈现人类智能的技术。该词也指出研究这样的智能系统是否能够实现，以及如何实现。同时，通过医学、神经科学、机器人学及统计学等的进步，常态预测则认为人类的很多职业也逐渐被其取代。

Compare the opening definition of machine intelligence with the final prediction about
human occupations. Does the second model response preserve both meanings? Keep the last
claim as a prediction, including its uncertainty; do not turn it into an established
fact, a claim about every occupation, or a precise deadline.

**B — your comparison:** describe how the English preserves the definition and prediction,
or identify one change in meaning.

If Ollama is unavailable, compare your predictions with the checked explanation below.

In [ ]:
translation_sources = {
    "Google Translate (historical statistics)": (
        "Google Translate支持249种语言。日均用户超过2亿人，2016年4月总用户数超过5亿人，"
        "每天翻译超过1000亿个单词。"
    ),
    "AI definition (original passage)": (
        "人工智能亦称智械、机器智能，指由人制造出来的机器所表现出来的智能。"
        "通常人工智能是指通过普通计算机程序来呈现人类智能的技术。"
        "该词也指出研究这样的智能系统是否能够实现，以及如何实现。"
        "同时，通过医学、神经科学、机器人学及统计学等的进步，"
        "常态预测则认为人类的很多职业也逐渐被其取代。"
    ),
}
original_translations = {}
for title, source_text in translation_sources.items():
    print("\nSource:", title, "\n", source_text)
    original_translations[title] = generate(
        "Translate into English. Return only the translation:\n" + source_text,
        max_tokens=256,
    )

**Check A — numbers and dates:**

| Source expression | Meaning to preserve in English |
| --- | --- |
| 249种语言 | 249 languages |
| 日均用户超过2亿人 | More than 200 million daily users |
| 2016年4月 | April 2016 |
| 总用户数超过5亿人 | Over 500 million total users |
| 每天翻译超过1000亿个单词 | Over 100 billion words translated per day |

One **亿** is 100 million, so **2亿 = 200 million**, **5亿 = 500 million**, and
**1000亿 = 100 billion**. Keep daily and total users distinct; retain the April 2016
date for total users. An English sentence can be fluent while changing a number,
unit, qualifier, or time period.

Compare with `original_translations["Google Translate (historical statistics)"]` from
the existing call above. Different English wording can preserve the same meaning.

**Check B — definition and prediction:**

- The opening defines artificial or machine intelligence as intelligence exhibited by
  human-made machines, then describes computer programs presenting human intelligence.
  It also refers to studying whether such systems can be built and how.
- The final sentence reports a prediction that many human occupations may gradually be
  replaced as related technologies advance. Preserve its status as a prediction and
  its uncertainty; “many” does not mean “all,” and no replacement deadline is given.
- Translating the prediction faithfully does not establish that it will come true.
  Check the whole passage for omitted qualifications or added claims.

Compare with `original_translations["AI definition (original passage)"]` from the same
cell. The A and B comparisons use the two existing responses, with the same model and
decoding settings.

### Article generation: human or machine?

These are the complete excerpts shown on [the original article-generation slide](https://baojian.github.io/llm-26/slides/lecture-01-slides/index.html#/8).
Read both before revealing the prior deck's answer. These are historical examples, not
outputs generated by the Qwen calls in this notebook; the news passage is not a verified
report of events.

**A · Blog**  
**Title: Feeling unproductive? Maybe you should stop overthinking.**

> In order to get something done, maybe we need to think less. Seems counter-intuitive,
> but I believe sometimes our thoughts can get in the way of the creative process.
> We can work better at times when we "tune out" the external world and focus on what's
> in front of us. I've been thinking about this lately, so I thought it would be good
> to write an article about it…

**B · News article**  
**Title: United Methodists Agree to Historic Split**

> After two days of intense debate, the United Methodist Church has agreed to a historic split —
> one that is expected to end in the creation of a new denomination, one that will be
> “theologically and socially conservative,” according to The Washington Post. The majority of
> delegates attending the church‘s annual General Conference in May voted to strengthen a ban on
> the ordination of LGBTQ clergy and to write new rules that will “discipline” clergy who officiate
> at same-sex weddings. But those who opposed these measures have a new plan…

Choose one: **1)** A: human, B: human; **2)** A: machine, B: human;
**3)** A: human, B: machine; **4)** A: machine, B: machine.

**Your choice and evidence:** replace this sentence before revealing the answer.

<details>
<summary>Reveal the original deck's answer</summary>

The prior deck attributes **both excerpts to GPT-3** (choice 4). This is the attribution
reported by that teaching source; this notebook does not independently establish their
generation provenance. Fluency alone is weak evidence about authorship or factual accuracy.

</details>

### Optional additional translation and generation

The core E02 cell already translates the full A/B passages. Leave the switch below off
during the timed exercise. After the authorship discussion, enable it to compare a short
AI-definition translation and a new generated paragraph. These would be new local-model
outputs, separate from the historical GPT-3 examples above.

In [ ]:
RUN_EXTRA_GENERATION = False
if RUN_EXTRA_GENERATION:
    chinese_text = (
        "人工智能亦称机器智能，指由人制造出来的机器所表现出来的智能。"
        "通常人工智能是指通过计算机程序来呈现人类智能的技术。"
    )
    translation = generate("Translate into English. Return only the translation:\n"
                           + chinese_text, max_tokens=128)
    welcome = generate(
        "Write a two-sentence welcome for Fudan's course on NLP and LLMs. "
        "Mention today's topic: tokenization. Do not invent course requirements.",
        max_tokens=96,
    )
else:
    print("Extra translation and generation are optional; enable the switch to run.")

### Task 4 · Image understanding with Qwen3-VL

Use **`qwen3-vl:2b`** to describe the Big Data illustration from
[the original image-question slide](https://baojian.github.io/llm-26/slides/lecture-01-slides/index.html#/9).
The earlier `qwen3:0.6b` text model cannot process images.

### Big Data image

<img src="assets/vision-big-data.png" alt="Big Data illustration with a central label, surrounding technology logos, and connecting lines" width="500">

Before running the model, inspect the [full-size image](assets/vision-big-data.png):

- Which labels and logos can you identify?
- What do the connecting lines visibly join?
- Which claims about those connections would require evidence beyond the image?

**Your observations:** record your answers, then compare them with the model response.

### Prepare and run

With Ollama installed and running, prepare the vision model **once in a terminal**:

```sh
ollama pull qwen3-vl:2b
```

Download before class and check that your computer can run it. Run the notebook's
imports and Ollama client setup above. Then **run both vision code cells below in order**:
first define `vision_payload`, then set **`RUN_VISION = True`** in the request cell,
and run it. Keep
`RUN_OLLAMA` enabled for local requests. The cell uses the bundled image and checks that
the model is installed and reports vision support; it does not download a model.

The REST API takes base64 image data in `messages[].images`. For `/api/chat`, the answer
is in `message.content`. The [`qwen3-vl:2b`](https://ollama.com/library/qwen3-vl:2b-thinking)
tag uses the Thinking variant: its generation budget includes thinking before the answer,
and [`think=False` cannot disable that behavior](https://github.com/ollama/ollama/issues/16945#issuecomment-4825564941).
The helper follows Qwen's [sampling settings](https://huggingface.co/Qwen/Qwen3-VL-2B-Thinking#generation-hyperparameters)
with a fixed seed and allows **2,048 generated tokens** shared by thinking and the answer.
The vision request can take several minutes and has a five-minute timeout. If the response
hits the token limit, increase `num_predict` in `vision_payload` and rerun both vision code
cells. This demonstration is optional and does not add a timed exercise.

<details>
<summary>Check the image after comparing your response</summary>

The illustration contains a central Big Data label, surrounding technology logos, and
connecting lines. Distinguish those visible elements from interpretations of what the
connections mean. The illustration alone does not establish commercial relationships
or data sharing.

</details>

In [ ]:
def vision_payload(image_bytes, model, prompt="Describe only what is visible in this image in two sentences."):
    # The token budget includes thinking and the visible answer.
    return {
        "model": model, "stream": False,
        "messages": [{"role": "user", "content": prompt,
                      "images": [base64.b64encode(image_bytes).decode("ascii")]}],
        "options": {"temperature": 1.0, "top_p": 0.95, "top_k": 20,
                    "seed": 42, "num_predict": 2048},
    }

In [ ]:
RUN_VISION = False  # Set True after preparing qwen3-vl:2b in Ollama.
VISION_MODEL = "qwen3-vl:2b"
IMAGE_PATH = Path("assets/vision-big-data.png")
VISION_PROMPT = (
    "Describe the image in one paragraph. Identify the central label, surrounding "
    "logos, and connecting lines. Separate visible content from inferred meaning."
)
if RUN_VISION and RUN_OLLAMA:
    installed = {m["name"] for m in ollama_request("/api/tags")["models"]}
    if VISION_MODEL not in installed:
        print("Install a vision model first; no model was downloaded.")
    elif not IMAGE_PATH.is_file():
        print("Set IMAGE_PATH to an existing PNG or JPEG image.")
    else:
        info = ollama_request("/api/show", {"model": VISION_MODEL})
        if "vision" not in info.get("capabilities", []):
            raise ValueError("The selected model does not report vision capability.")
        print(f"Running {VISION_MODEL} on the Big Data image; allow up to five minutes...")
        response = ollama_request(
            "/api/chat", vision_payload(IMAGE_PATH.read_bytes(), VISION_MODEL, VISION_PROMPT),
            timeout=300,
        )
        answer = response["message"]["content"].strip()
        if answer:
            print(answer)
        else:
            print("No visible answer was returned; check the generation budget.")
        if response.get("done_reason") == "length":
            print("Generation reached the token limit; the answer may be incomplete.")
            print("Increase num_predict in vision_payload and rerun both vision code cells.")

### Text → image: Stable Diffusion

The [original text-to-image demo](https://baojian.github.io/llm-26/slides/lecture-01-slides/index.html#/10)
starts with this prompt:

> a watercolor painting of a university campus gate at sunset, people fully clothed, family-friendly

Predict which parts a generated image should depict. Check objects, style, lighting, and
whether any requested detail is missing. This notebook can generate a new sample from
that prompt using **an already prepared local Stable Diffusion pipeline**.

Optional setup, in a terminal at the repository root:

```sh
uv sync --extra tokenization --extra multimodal
```

Prepare a compatible Stable Diffusion model separately before class, then set `SD_MODEL`
to its complete local pipeline directory or its already-cached model ID. A complete
Diffusers pipeline includes its configuration, tokenizer, text encoder, denoiser, VAE,
and other configured components. Installing Python packages does not cache model files.
The example defaults to the Stable Diffusion 1.5 model ID, but **only cached files are
loaded**; missing files produce a message, not a model download.

Set `RUN_DIFFUSION = True` only when ready. Choose `cpu`, `cuda`, or `mps` for your
hardware; CPU generation can be slow. The loader keeps the pipeline's configured safety
components. A fixed seed helps comparisons, but results may differ across devices and
software versions. See [Diffusers pipeline loading](https://huggingface.co/docs/diffusers/api/pipelines/overview)
for `local_files_only`.

In [ ]:
def generate_local_image(model, prompt, *, device="cpu", seed=42, steps=30, guidance_scale=7.5):
    """Generate from existing local/cached model files; never fetch model files."""
    import torch
    from diffusers import DiffusionPipeline

    dtype = torch.float16 if device == "cuda" else torch.float32
    pipeline = DiffusionPipeline.from_pretrained(
        model, local_files_only=True, use_safetensors=True, dtype=dtype,
    )
    pipeline = pipeline.to(device)
    generator = torch.Generator(device="cpu").manual_seed(seed)
    result = pipeline(
        prompt=prompt, num_inference_steps=steps, guidance_scale=guidance_scale,
        generator=generator,
    )
    return result.images[0]

In [ ]:
RUN_DIFFUSION = False
SD_MODEL = os.environ.get("COURSE_SD_MODEL", "stable-diffusion-v1-5/stable-diffusion-v1-5")
DIFFUSION_DEVICE = "cpu"  # Use "cuda" or "mps" only when available on your computer.
DIFFUSION_PROMPT = (
    "a watercolor painting of a university campus gate at sunset, "
    "people fully clothed, family-friendly"
)
if RUN_DIFFUSION:
    try:
        from IPython.display import display

        generated_image = generate_local_image(SD_MODEL, DIFFUSION_PROMPT, device=DIFFUSION_DEVICE)
        display(generated_image)
    except (ImportError, OSError, ValueError) as error:
        print("Image generation unavailable:", error)
        print("Prepare optional packages and a complete local/cached model first. "
              "No model files were downloaded.")

### Text → video: inspect the Johannesburg clip

This recorded example comes from [the original text-to-video slide](https://baojian.github.io/llm-26/slides/lecture-01-slides/index.html#/11).
The notebook **plays an existing clip**; it does not run a video-generation model.

[Play the Johannesburg clip](assets/text-to-video-johannesburg.mp4). Prompt:

> a woman wearing purple overalls and cowboy boots taking a pleasant stroll in Johannesburg South Africa during a beautiful sunset

Check prompt fidelity and consistency across frames: clothing, scenery, body motion,
and whether details change unexpectedly. The prompt describes a requested scene;
the clip is not evidence of real events at that location.

Use the link above, or set `RUN_VIDEO_PLAYBACK = True` below to embed a player in the
personal notebook. Embedding includes the local video bytes in notebook output.

In [ ]:
RUN_VIDEO_PLAYBACK = False
VIDEO_PATH = Path("assets/text-to-video-johannesburg.mp4")
if RUN_VIDEO_PLAYBACK:
    from IPython.display import Video, display

    if VIDEO_PATH.is_file():
        display(Video(filename=str(VIDEO_PATH), embed=True, width=640,
                      html_attributes="controls muted loop"))
    else:
        print("Missing local clip:", VIDEO_PATH)

### Course website and GPU resources

[baojian.github.io/llm-26-fall/](https://baojian.github.io/llm-26-fall/)

- Weekly schedule, slides, and notebooks.
- Assignments, project requirements, and grading.
- Course policies and learning resources.

**Fudan Qizhi GPU platform:** [qz.cfff.fudan.edu.cn](http://qz.cfff.fudan.edu.cn/)

- Register through [Fudan CFFF](https://cfff.fudan.edu.cn/home).
- Use GPU training and inference for coursework and projects.
- Per-student GPU and AI-tool budgets will be specified with the project handout.
  Exact model sizes and run budgets follow staff pilots; see the
  [course computing guidance](https://baojian.github.io/llm-26-fall/#resources).

The core Lecture 01 preprocessing and BPE exercises run locally without a GPU.

**Bookmark both sites and check for updates.**

<a id="development"></a>
## 2. Development of NLP & LLMs

The section has **ten historical slides followed by three September 2026 updates**.
Dates identify selected papers, not sharp boundaries between methods. Rules, statistical models, and neural systems still coexist.

| Page | Date / topic | Change to understand | Primary source |
| --- | --- | --- | --- |
| 1 | 1949 · Weaver | Use context and statistical patterns to study translation | [Translation memorandum](https://mt-archive.net/Weaver-1949.pdf) |
| 2 | 1950 · Turing test | Evaluate anonymous text interaction | [Turing, §§1–2](https://www.csee.umbc.edu/courses/471/papers/turing.pdf) |
| 3 | 1950 · Evidence | Human-like replies and correct answers require different checks | [Turing, §2](https://www.csee.umbc.edu/courses/471/papers/turing.pdf) |
| 4 | 1960s–1990s · Rules and statistics | Move from explicit patterns to estimated probabilities | [ELIZA](https://doi.org/10.1145/365153.365168); [Brown et al.](https://aclanthology.org/J90-2002/) |
| 5 | 2003–2014 · Neural representations | Learn vectors and sequence mappings | [Bengio et al.](https://www.jmlr.org/papers/v3/bengio03a.html); [word2vec](https://arxiv.org/abs/1301.3781); [seq2seq](https://arxiv.org/abs/1409.3215) |
| 6 | 2014–2017 · Attention | Access source positions; then remove recurrence from the Transformer | [Bahdanau et al.](https://arxiv.org/abs/1409.0473); [Transformer](https://arxiv.org/abs/1706.03762) |
| 7 | 2018 · Pretraining | Learn from text before task adaptation | [BERT](https://arxiv.org/abs/1810.04805); [GPT](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf) |
| 8 | 2020 · In-context learning | Specify tasks in the prompt, with fixed weights | [GPT-3](https://arxiv.org/abs/2005.14165) |
| 9 | 2022 · Instruction tuning | Demonstrations and preferences change model behavior | [InstructGPT](https://arxiv.org/abs/2203.02155) |
| 10 | 2023–2025 · Access, reasoning, tools | Run models; evaluate reasoning; use external observations | [LLaMA](https://arxiv.org/abs/2302.13971); [DeepSeek-R1](https://arxiv.org/abs/2501.12948); [ReAct](https://arxiv.org/abs/2210.03629) |

ReAct's preprint is from 2022 and conference publication from 2023. BERT's preprint is
from 2018 and conference publication from 2019. Attention for translation predates the
Transformer. The original Transformer figure includes an encoder and decoder.

![Simplified Turing test](assets/turing-test-rooms.png)

*AI-generated conceptual illustration. The judge cannot see the identities visible to us.
This is the familiar machine-versus-human adaptation of Turing's original imitation game.*

**Discuss:** a judge accepts a reply as human. What evidence do we still need before using
that reply as a translation, a calculation, or a statement of fact?

Turing's paper replies **105,621** to **34,957 + 70,764**; the checked sum is **105,721**.
This is a historical example, not a recorded result from our Qwen model.

**Your observation:** choose one history page and explain what changed and what remained
hard. Does adding another example to a prompt change the model's weights? **Check:** no;
prompting changes the context, whereas fine-tuning updates weights.

The [original survey timeline](assets/llm-timeline-2019-2024.png) covers selected models
from **2019–2024** (2025 revision). It is a historical reference, not a current model
ranking or a genealogy. Available weights, code, data, and licenses are different artifacts.

### September 2026: models and scientific research

Snapshot: **September 9, 2026**. These three updates follow the ten historical slides.

| Developer | Model / family | Official reading |
| --- | --- | --- |
| Anthropic | Claude Fable 5.1 | [Model overview](https://platform.claude.com/docs/en/models/fable-5-1/overview) |
| OpenAI | GPT-6 Astra | [Model documentation](https://developers.openai.com/api/docs/models/gpt-6-astra) |
| Z.ai | GLM-5.3 | [Model card and weights](https://huggingface.co/zai-org/GLM-5.3) |
| Alibaba | Qwen3.8 | [Qwen3.8-Flash-Next release](https://qwen.ai/blog?id=qwen3.8-flash-next) |
| Moonshot AI | Kimi K3 | [Official repository](https://github.com/MoonshotAI/Kimi-K3) |
| DeepSeek | DeepSeek-V4-Pro | [Model card](https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro); [August 13 update](https://api-docs.deepseek.com/news/news260813/) |

GPT-6 Astra is a model name; ChatGPT is an application. Qwen3.8 names a family;
the linked release describes a particular variant. Use the prepared Qwen model for
today's classroom calls; these readings do not require installing another model.

**Terminal-Bench Science.** The [0.1 announcement](https://www.terminal-bench-science.ai/announcement) describes
70 tasks across five scientific domains, with checks on concrete research outputs.
See the [Fable 5.1 announcement](https://www.anthropic.com/claude-fable-and-mythos-5-1)
and [GPT-6 Astra announcement](https://openai.com/index/gpt-6-astra/) for their evaluations.
The [contribution guide](https://www.terminal-bench-science.ai/contribute) explains how
researchers can propose new tasks.

[Follow the leaderboard](https://www.terminal-bench-science.ai/) and
[browse the tasks](https://github.com/harbor-framework/terminal-bench-science).
Choose one task and identify the required output and its verification method.
Compare results only after checking the task version, agent software, tools, and trial
conditions. Following the project is optional; it adds no graded requirement.

**Navier–Stokes research announcement.** On September 8, OpenAI
[reported a solution](https://openai.com/index/navier-stokes-solution/) using an internal
model stronger than GPT-6 Astra and a system of collaborating agents.
The [paper, Theorem 1.1](https://cdn.openai.com/pdf/32d9f210-8b73-45e0-91bc-82a30aef8a9a/navier-stokes.pdf)
claims finite-time blowup for smoothly forced three-dimensional flow with finite
kinetic energy. A [Lean formalization](https://github.com/openai/NavierStokesAndEuler)
is available for inspection.

As of this snapshot, [Clay lists the problem as unsolved](https://www.claymath.org/millennium/navier-stokes-equation/).
Distinguish the reported result, what a formal checker verifies, and independent
assessment of the mathematical statement. The
[news discussion shared in class](https://x.com/i/trending/2097205038976524548)
is a starting point; read the primary evidence above.


In [ ]:
turing_sum = 34957 + 70764
assert turing_sum == 105721
print("Checked sum:", turing_sum)
print("Difference from the paper's sample reply:", turing_sum - 105621)

<a id="preprocessing"></a>
## 3. Basics for Text Preprocessing

A **corpus** is a collection of texts. Books, news, reviews, forums, code, and documentation
differ in language, style, topics, repetition, and formatting. Inspect your sources before
choosing a processing rule. Our BPE corpora are **small synthetic teaching examples**.

Conceptually: **decode bytes → choose normalization → define chunks → encode token IDs**.
Each stage has a different job. Splitting text and deleting text are different operations.

### Preserve the text you intend to model

Inspect the two spaces, newline, and capitalization in the following example.
Lowercasing changes the input; `strip()` removes whitespace only at its ends.

Consider the cost of deleting `not` from a review, lowercasing `US`, deleting the dot in
`3.14`, or collapsing indentation in Python. A transformation suitable for one task can
harm another.

In [ ]:
text = "Hello,  世界!\n🙂"
print(repr(text))
print(repr(text.lower().strip()))
assert text != text.lower().strip()
assert text.strip() == text

### Regular expressions: explicit matching rules

A regex describes a pattern; it does not understand a sentence. Python's built-in `re`
works with Unicode strings. Use a raw Python string such as `r"[wW]oodchuck"`.

| Pattern | Meaning |
| --- | --- |
| `[wW]oodchuck` | Either lowercase or uppercase initial w |
| `[A-Z]` | One ASCII uppercase letter |
| `[^0-9]` | One character that is not an ASCII digit |
| `cat\|dog` | Either alternative (the pattern itself uses an unescaped pipe) |
| `colou?r` | Optional u |
| `oh*` / `oh+` | Zero-or-more / one-or-more h characters |
| `beg.n` | One character between g and n; newline excluded by default |

`search` finds the first match or returns `None`. `findall` collects non-overlapping
matches; capturing groups affect its return value. `fullmatch` checks the entire string.
The exercises use a noncapturing group `(?:...)` when collecting complete words.

Source: [Python re documentation](https://docs.python.org/3/library/re.html), syntax and APIs.

In [ ]:
import re

assert re.findall(r"[wW]oodchuck", "Woodchuck and woodchuck") == ["Woodchuck", "woodchuck"]
assert re.findall(r"[^0-9]", "A 3你") == ["A", " ", "你"]
assert re.findall(r"cat|dog", "cat and dog") == ["cat", "dog"]
assert re.findall(r"colou?r", "color colour") == ["color", "colour"]
assert re.fullmatch(r"oh*", "o") is not None
assert re.fullmatch(r"oh+", "o") is None
assert re.search(r"beg.n", "begin begun").group() == "begin"
assert re.fullmatch(r"beg.n", "beg\nn") is None
print("All regex examples checked.")

### P01 · What did the pattern discard? · 2 minutes

Use `r"[A-Za-z]+(?:'[A-Za-z]+)?"` on **`Senjō 3 can't 你好🙂`**.
Predict the extracted strings, then explain whether the exact input can be reconstructed.

**Your prediction and explanation:** write here before running.

This short example isolates the same problem as the Spring notebook's full Valkyria
passage: an ASCII-letter extractor drops text outside its pattern.

In [ ]:
regex_text = "Senjō 3 can't 你好🙂"
word_pattern = r"[A-Za-z]+(?:'[A-Za-z]+)?"
extracted_words = re.findall(word_pattern, regex_text)
print(extracted_words)
assert extracted_words == ["Senj", "can't"]
assert "".join(extracted_words) != regex_text

**Check P01:** `['Senj', "can't"]`. The pattern loses ō, 3, Chinese characters, emoji,
and spaces. The apostrophe group keeps `can't` together. The result is useful for a
specified extraction task, but cannot reconstruct this exact input.

### Preserve every character, then question the boundaries

The next pattern covers whitespace, word characters, and every remaining character.
It is a **toy partition**, not a production tokenizer or a Chinese word segmenter.

In [ ]:
regex_chunks = re.findall(r"\s+|\w+|[^\w\s]", regex_text)
print(regex_chunks)
assert "".join(regex_chunks) == regex_text
assert "你好" in regex_chunks
assert "'" in regex_chunks
assert re.findall(r"\w+", "我们学习语言模型") == ["我们学习语言模型"]

### Language still needs context

The Spring lecture's examples remain useful discussion prompts:

- **A man saw a boy with a telescope.** Who had the telescope?
- **Mighty Dragon.** Which referent does the surrounding text establish?
- **He has quit smoking.** This normally presupposes earlier smoking.
- **冬天，能穿多少穿多少；夏天，能穿多少穿多少。** The winter reading is to wear as much
  as possible; the summer reading is to wear as little as possible.
- **A penny is better than nothing; nothing is better than world peace.** The two uses
  of “nothing” differ, so the apparent inference that a penny is better than peace fails.

Nonstandard spelling, hashtags, emojis, idioms such as “break a leg,” and new words
make universal cleanup rules difficult. These are language-analysis examples, not claims
that an untested model always fails them.

### Unicode and UTF-8

A Unicode code point is a character value; a **grapheme cluster** can contain multiple
code points and often corresponds to a user-perceived character. UTF-8 encodes a Unicode
scalar value in one to four bytes. Python `len` counts code points in these strings.
JavaScript `string.length` counts UTF-16 code units, so the slide demo uses `Array.from`.

The live browser demo and the following exercise use the same inputs.

### E03 · Code points, bytes, and normalization · 3 minutes

Predict `len(text)` and `len(text.encode("utf-8"))` for the examples below.
The two accented strings look alike; the second uses `e` followed by a combining accent.

**Your predictions:** replace this sentence before running the cell.

In [ ]:
import unicodedata

examples = ["hello", "你好", "🙂", "é", "e\u0301", "你好🙂"]
for text in examples:
    print(repr(text), "code points:", len(text),
          "UTF-8 bytes:", len(text.encode("utf-8")))
    assert text.encode("utf-8").decode("utf-8") == text

assert "é" != "e\u0301"
assert unicodedata.normalize("NFC", "e\u0301") == "é"
print("NFC can make these strings equal, but changes the original representation.")

**Check:** `hello` gives 5 / 5; `你好` 2 / 6; `🙂` 1 / 4; `é` 1 / 2;
`e\u0301` 2 / 3; `你好🙂` 3 / 10. These are code points and bytes, not model-token counts.
Our tokenizer preserves the exact input; it does not lowercase or normalize it.

In [ ]:
normalization_examples = [("NFC", "e\u0301"), ("NFKC", "Ａ")]
for form, original in normalization_examples:
    normalized = unicodedata.normalize(form, original)
    print(form, repr(original), "->", repr(normalized))
assert unicodedata.normalize("NFC", "e\u0301") == "é"
assert unicodedata.normalize("NFKC", "Ａ") == "A"
assert "US".lower() == "us"
# Lowercasing is a separate operation, not a Unicode normalization form.

In [ ]:
text = "你好🙂"
byte_ids = list(text.encode("utf-8"))
print("Byte IDs:", byte_ids)
print("Full round trip:", bytes(byte_ids).decode("utf-8"))
try:
    print(bytes(byte_ids[:1]).decode("utf-8"))
except UnicodeDecodeError:
    print("One byte of this character is not a complete UTF-8 character.")
assert bytes(byte_ids).decode("utf-8") == text

<a id="tokenization"></a>
## 4. Text Tokenization

A tokenizer maps text to **token IDs**. A model looks up an embedding vector for each ID.
Token IDs belong to one vocabulary; our toy IDs must not be supplied to a pretrained Qwen
model. A token is not necessarily a whole word, morpheme, or complete UTF-8 character.

An autoregressive language model defines probabilities through

$$p_\theta(t_1,\ldots,t_L)=\prod_{i=1}^{L}p_\theta(t_i\mid t_{<i}).$$

Generation selects a next token, appends it to the context, and repeats. This is a
conditional factorization, not an independence assumption. Likelihood does not guarantee
factual accuracy. The earlier local-model calls combine this mechanism with a tokenizer,
a model runtime, and prompt formatting.

| Basic unit | Benefit | Limitation |
| --- | --- | --- |
| Word | Familiar units | Unseen words and ambiguous boundaries |
| Code point | Direct character values | Large alphabet or unknown values in a learned vocabulary |
| Byte | Only 256 base values | Long sequences |
| Learned subword | Share frequent pieces | Coverage and segmentation depend on training and rules |

### P02 · Tokens and types · 2 minutes

For **`low low lower`**, use whitespace-separated words. Count tokens (all occurrences)
and types (distinct forms), then predict the byte-token count including spaces.

**Your prediction:** write here.

In [ ]:
counting_text = "low low lower"
word_tokens = counting_text.split()
word_types = set(word_tokens)
print("Word tokens:", len(word_tokens), "Word types:", len(word_types))
print("Byte tokens:", len(counting_text.encode("utf-8")))
assert (len(word_tokens), len(word_types)) == (3, 2)
assert len(counting_text.encode("utf-8")) == 13

**Check P02:** 3 word tokens, 2 word types, and 13 byte tokens. Always name the units
and the segmentation rules when reporting a corpus size or a context length.

**Why subwords?** A vocabulary with `low`, `er`, and `est` could reuse pieces across
`low`, `lower`, and unseen `lowest`. This is an illustration, not the vocabulary of a
named tokenizer. Our two-merge example will learn `low` and encode `lowest` as
`[low, e, s, t]` because it does not learn `est`.

### Train a byte BPE tokenizer

We retain all 256 base bytes. Each merge creates one additional token whose value is
the concatenation of two existing byte strings. No unknown token is needed for unseen
valid UTF-8 text. Token IDs are specific to this vocabulary and cannot be passed to Qwen.

This teaching implementation follows the training/encoding distinction in
[CS336 Lecture 1](https://cs336.stanford.edu/lectures/?trace=lecture_01).
It scans sequences repeatedly to keep the mechanism visible. It has no production
pre-tokenizer, special-token parser, or chat template. Each input string is a separate
sequence; merges never cross between strings.

**Tie rule:** among pairs with the highest count, choose the smallest tuple of token IDs.

In [ ]:
def count_pairs(sequences):
    """sequences contains (token_ids, frequency); count overlaps within each one."""
    counts = Counter()
    for ids, frequency in sequences:
        for pair in zip(ids, ids[1:]):
            counts[pair] += frequency
    return counts


def merge_pair(ids, pair, new_id):
    """Replace non-overlapping occurrences from left to right."""
    result = []
    index = 0
    while index < len(ids):
        if tuple(ids[index:index + 2]) == pair:
            result.append(new_id)
            index += 2
        else:
            result.append(ids[index])
            index += 1
    return result

In [ ]:
def train_bpe(corpus, num_merges):
    """Return vocabulary, ordered merges, and weighted token totals."""
    if type(num_merges) is not int or num_merges < 0:
        raise ValueError("num_merges must be a non-negative integer")
    sequences = []
    for text, frequency in corpus:
        if not isinstance(text, str) or type(frequency) is not int or frequency <= 0:
            raise ValueError("Each corpus item needs text and a positive integer frequency")
        sequences.append((list(text.encode("utf-8")), frequency))
    vocab = {index: bytes([index]) for index in range(256)}
    merges = []
    totals = [sum(len(ids) * frequency for ids, frequency in sequences)]
    for _ in range(num_merges):
        counts = count_pairs(sequences)
        if not counts:
            break
        pair = min(counts, key=lambda pair: (-counts[pair], pair))
        new_id = len(vocab)
        vocab[new_id] = vocab[pair[0]] + vocab[pair[1]]
        merges.append((pair, new_id))
        sequences = [(merge_pair(ids, pair, new_id), frequency)
                     for ids, frequency in sequences]
        totals.append(sum(len(ids) * frequency for ids, frequency in sequences))
    return vocab, merges, totals

In [ ]:
class ByteBPETokenizer:
    def __init__(self, vocab, merges):
        self.vocab = dict(vocab)
        self.merges = list(merges)

    def encode(self, text):
        ids = list(text.encode("utf-8"))
        for pair, new_id in self.merges:
            ids = merge_pair(ids, pair, new_id)
        return ids

    def decode(self, ids):
        return b"".join(self.vocab[index] for index in ids).decode("utf-8")

    def pieces(self, text):
        return [self.vocab[index] for index in self.encode(text)]

### Reproduce the lecture's two merges

Treat `low` and `lower` as separate sequences. The first tie is between `(l, o)` and
`(o, w)`; our rule picks `(l, o)`. Because these letters are ASCII, character and byte
counts coincide here. The chart in the slides uses exactly these weighted totals.

In [ ]:
toy_corpus = [("low", 5), ("lower", 2)]
vocab, merges, totals = train_bpe(toy_corpus, 2)
tokenizer = ByteBPETokenizer(vocab, merges)
for pair, new_id in merges:
    print(pair, "->", new_id, repr(vocab[new_id]))
print("Weighted token totals:", totals)
print("Vocabulary size:", len(vocab))
assert totals == [25, 18, 11]
assert merges == [((108, 111), 256), ((256, 119), 257)]
assert len(vocab) == 258

for sample, frequency in toy_corpus:
    print(sample, "x", frequency, "->", tokenizer.pieces(sample))

### E04 · Overlapping pairs · 4 minutes

Training corpus: `aaab` × 2 and `ab` × 1. Which pair wins? Predict the weighted
total token count after one merge. Remember that counting and replacement handle
overlaps differently.

**Your counts and predicted total:** replace this sentence before running the check.

In [ ]:
overlap_corpus = [("aaab", 2), ("ab", 1)]
sequences = [(list(text.encode("utf-8")), frequency)
             for text, frequency in overlap_corpus]
counts = count_pairs(sequences)
print({bytes(pair): count for pair, count in counts.items()})
overlap_vocab, overlap_merges, overlap_totals = train_bpe(overlap_corpus, 1)
overlap_tokenizer = ByteBPETokenizer(overlap_vocab, overlap_merges)
print("After one merge:", overlap_tokenizer.pieces("aaab"))
print("Weighted totals:", overlap_totals)
assert counts[(97, 97)] == 4 and counts[(97, 98)] == 3
assert overlap_totals == [10, 8]

**Check:** `aa` has four adjacent occurrences, but only two non-overlapping replacements
across the weighted corpus. Each `aaab` becomes `[aa, a, b]`; `ab` is unchanged.
The weighted total falls from 10 to 8, not to 6.

### E05 · Encode unseen text · 5 minutes

Use the **fixed** vocabulary and merge order learned above. Predict the token IDs for
`lowest`, then predict whether `你好🙂` can still be represented exactly.

**Your predictions:** replace this sentence before running the check.

In [ ]:
for text in ["lowest", "你好🙂", "low low", "", "e\u0301"]:
    ids = tokenizer.encode(text)
    print(repr(text), ids, tokenizer.pieces(text))
    assert tokenizer.decode(ids) == text
assert tokenizer.encode("lowest") == [257, 101, 115, 116]
assert len(tokenizer.encode("你好🙂")) == 10
assert len(tokenizer.vocab) == 258  # Encoding did not add new entries.

**Check:** `lowest` becomes `[low, e, s, t]`. The Chinese/emoji string has ten base-byte
tokens because it contains neither learned ASCII pair. It still round-trips exactly.
An empty string becomes an empty token list. Training learns the vocabulary; encoding
does not grow it.

### Merge rank beats leftmost position

If `(b, c)` was learned before `(a, b)`, encoding `abc` yields `[a, bc]`. A longest-prefix
lookup could instead choose `[ab, c]`, which is a different algorithm. Verify with a
training corpus that makes those ranks occur.

In [ ]:
rank_vocab, rank_merges, _ = train_bpe([("bc", 3), ("ab", 2)], 2)
rank_tokenizer = ByteBPETokenizer(rank_vocab, rank_merges)
print("Merge order:", [rank_vocab[new_id] for _, new_id in rank_merges])
print("abc:", rank_tokenizer.pieces("abc"))
assert rank_tokenizer.pieces("abc") == [b"a", b"bc"]

### The larger Spring corpus and word boundaries

The original worked corpus is `low`×5, `lowest`×2, `newer`×6, `wider`×3, `new`×2.
Append a toy end-of-word marker `_` to each separate sequence. This marker is reserved
only by convention in this ASCII exercise. The separate sequences prevent cross-word
merges; adding an underscore by itself is not a complete boundary policy.

The first two merges are `e r → er`, then `er _ → er_`, each with count 9. Later ties
can produce a different sequence from the Spring illustration; use the documented
smallest-token-ID rule. Each new piece must concatenate the bytes of its two parents.

In [ ]:
spring_words = [("low", 5), ("lowest", 2), ("newer", 6), ("wider", 3), ("new", 2)]
spring_corpus = [(word + "_", count) for word, count in spring_words]
spring_vocab, spring_merges, spring_totals = train_bpe(spring_corpus, 2)
spring_tokenizer = ByteBPETokenizer(spring_vocab, spring_merges)
print("Spring corpus totals:", spring_totals)
print("First pieces learned:", [spring_vocab[i] for _, i in spring_merges])
assert [spring_vocab[i] for _, i in spring_merges] == [b"er", b"er_"]
assert spring_totals == [96, 87, 78]
for word, count in spring_corpus:
    assert spring_tokenizer.decode(spring_tokenizer.encode(word)) == word
    print(word, "x", count, spring_tokenizer.pieces(word))

### Boundaries, special tokens, and algorithm families

Our core learner can merge anywhere **within** a training string, including across spaces.
It never merges across different strings. Production tokenizers often apply a
pre-tokenizer that restricts merge boundaries before the learned segmentation.

Chat systems additionally define roles and message boundaries through a **chat template**,
often with reserved special-token IDs. Typing a special token's spelling does not always
insert that reserved ID. Our ordinary-text toy BPE implements neither control-token parsing
nor a chat template.

| Method | Important distinction |
| --- | --- |
| BPE | Learn frequent adjacent merges; encode using saved merge ranks |
| WordPiece | Standard BERT encoding uses longest matching vocabulary pieces and continuation markers |
| Unigram | Learn piece probabilities and score alternative segmentations |

**SentencePiece is a toolkit**, supporting BPE and Unigram, not a fourth segmentation
objective. Its text handling and normalization also matter. The extended notebook contains
optional comparisons; only BPE is implemented in the core.

Sources: [BERT implementation](https://github.com/google-research/bert/blob/master/tokenization.py),
[Kudo (2018)](https://aclanthology.org/P18-1007/),
[SentencePiece](https://aclanthology.org/D18-2012/).

### Vocabulary and sequence length have different costs

A vocabulary of size $V$ and embedding width $d$ requires $Vd$ embedding entries.
Doubling sequence length from 1,000 to 2,000 increases the number of all-position pairs
from 1 million to 4 million. This illustrates full-attention work in training/prompt
processing; it does not claim every inference step or total runtime scales identically.

Compare tokenizer choices on the same held-out text and report preprocessing, vocabulary,
actual merges learned, token count by language, bytes per token, and round-trip success.

### E06 · Vocabulary size and held-out text · 8 minutes

Train toy tokenizers with 0, 8, and 32 merges. Hold the evaluation text fixed and keep it
out of training. These small, synthetic corpora illustrate the method; they are not a
language benchmark. Frequencies are teaching weights, not measured usage statistics.

For each language, report total tokens, UTF-8 bytes per token, and successful round trips.
Then repeat with **English-only** training. Which result changes, and why?

**Your prediction about English-only training:** replace this sentence before running.

In [ ]:
TRAIN_EN = [("the cat is on the mat", 4), ("the dog is in the room", 3),
            ("we study language models", 3), ("a model reads text", 2)]
TRAIN_ZH = [("我们学习语言模型。", 4), ("语言模型读取文本。", 3),
            ("这是一门课程。", 3), ("今天学习新的知识。", 2)]
HELD_OUT = {
    "English": ["the cat reads text", "we study a new model"],
    "Chinese": ["我们学习新的模型。", "这是一门语言课程。"],
}
training_texts = {text for text, _ in TRAIN_EN + TRAIN_ZH}
assert all(text not in training_texts for texts in HELD_OUT.values() for text in texts)

In [ ]:
def measure(tokenizer, texts):
    total_bytes = sum(len(text.encode("utf-8")) for text in texts)
    total_tokens = sum(len(tokenizer.encode(text)) for text in texts)
    roundtrip = all(tokenizer.decode(tokenizer.encode(text)) == text for text in texts)
    return {"tokens": total_tokens,
            "bytes_per_token": total_bytes / total_tokens if total_tokens else None,
            "roundtrip": roundtrip}


comparison_rows = []
for training_name, corpus in [("EN + ZH", TRAIN_EN + TRAIN_ZH), ("EN only", TRAIN_EN)]:
    for budget in [0, 8, 32]:
        current_vocab, current_merges, _ = train_bpe(corpus, budget)
        current = ByteBPETokenizer(current_vocab, current_merges)
        for language, texts in HELD_OUT.items():
            metrics = measure(current, texts)
            assert metrics["roundtrip"]
            row = {"training": training_name, "budget": budget,
                   "learned": len(current_merges), "vocab": len(current_vocab),
                   "language": language, **metrics}
            comparison_rows.append(row)
            print(training_name, "merges:", len(current_merges), language,
                  "tokens:", metrics["tokens"],
                  "bytes/token:", round(metrics["bytes_per_token"], 3))

**Check:** all inputs round-trip, including under English-only training. At zero merges,
bytes per token is exactly 1. English-only merges here cannot compress the Chinese
examples, whose UTF-8 bytes contain none of the learned ASCII pairs. Mixed training can
learn Chinese byte sequences. Inspect the measured counts instead of assuming equal
compression across languages.

**Discussion:**

- Does increasing the merge budget always help each language by the same amount?
- Why does lower token count not prove better model quality?
- What else must be held fixed to compare real model tokenizers fairly?

At embedding width 1,024, vocabularies of 32,000 and 64,000 require 32,768,000 and
65,536,000 embedding entries respectively. These are illustrative parameter counts.
Fewer tokens may reduce sequence-processing work, but larger vocabulary tables cost
memory. We have not trained a language model or measured downstream accuracy here.

### Controlled probes: spaces, digits, and normalization

Predict which changes alter the bytes or pieces. The outputs below describe our toy
vocabulary, not every pretrained model. Compression and model accuracy require separate
measurements.

In [ ]:
probe_texts = ["low", " low", "2026", "2,026", "é", "e\u0301"]
for probe in probe_texts:
    ids = tokenizer.encode(probe)
    print(repr(probe), "bytes:", len(probe.encode("utf-8")), "pieces:", tokenizer.pieces(probe))
    assert tokenizer.decode(ids) == probe

### Before you leave

1. Why can one visible symbol require several tokens?
2. What changes during BPE training, and what stays fixed during encoding?
3. Does a shorter token sequence prove that the model is more accurate?

**Your three-sentence answer:** write here.

**Check:** a visible symbol can comprise multiple code points and bytes; the tokenizer's
vocabulary determines how they are grouped. Training learns merges and vocabulary entries;
encoding keeps them fixed. Compression alone provides no downstream-accuracy result.

Next lecture: probabilities over token sequences and a first language-model baseline.

## Further experiments

The classroom core ends above. Task 4 image understanding is earlier in the application
gallery. Enable only the further experiment you want below. Model features
depend on the installed Ollama version and model; failed API calls report their actual
errors. Do not infer capabilities from a model family name alone.

### Token log probabilities

This recreates the local probability-inspection example from the previous notebook.
Ollama's current generate API accepts `logprobs` and `top_logprobs`. The returned
probability `exp(logprob)` is for a token conditioned on the prefix, **not** the probability
that the answer is factually correct. Top-k candidates need not sum to one. A token's
display text can be only part of a word; inspect the returned bytes if needed.

In [ ]:
RUN_LOGPROBS = False
if RUN_LOGPROBS:
    probability_response = generate(
        "Fudan University is located in which city? Answer with one word.",
        max_tokens=20, logprobs=True, top_logprobs=5,
    )
    if probability_response:
        entries = probability_response.get("logprobs", [])
        if not entries:
            print("No log probabilities returned; check runtime/model support.")
        for entry in entries:
            print(repr(entry["token"]), "p =", math.exp(entry["logprob"]))
            for alternative in entry.get("top_logprobs", []):
                print("  ", repr(alternative["token"]),
                      math.exp(alternative["logprob"]))

### Thinking output and final answer

For the Qwen3 models used here, `think=True` requests separate thinking output. Compare
it with the final `response`. The first prompt below restores the original Qwen demo's
Fudan-city question; compare it with `first_response`, which used `think=False`. The
second prompt is the decimal-comparison experiment.

The generated trace is not a guaranteed faithful account of the model's internal
computation. A short token budget may be exhausted before a final answer appears; an
empty final answer in that case does not mean the server failed.

In [ ]:
RUN_THINKING = False
thinking_prompts = [
    "Fudan University is located in which city? Answer with one word.",
    "Which is larger, 9.11 or 9.9? Explain briefly.",
]
if RUN_THINKING:
    for thinking_prompt in thinking_prompts:
        print("\nPrompt:", thinking_prompt)
        thinking_response = generate(thinking_prompt, think=True, max_tokens=256)
        if thinking_response:
            print("Thinking:", thinking_response.get("thinking", "[not returned]"))
            print("Final answer:", thinking_response.get("response", ""))
            print("Stop reason:", thinking_response.get("done_reason"))

### Compare named tokenizer encodings

`tiktoken` is already in the course environment. Its first use of an encoding may download
that encoding's vocabulary; this optional cell therefore needs network access unless
already cached. `cl100k_base` and `o200k_base` are **encoding names**, not evidence about
which tokenizer an unverified model version uses.

Inspect spaces, digits, Chinese, and emoji. Individual token byte strings may not decode
as standalone UTF-8. `encode_ordinary` treats text literally rather than interpreting
special-token spellings as control IDs. Record the package version and encoding name.

In [ ]:
RUN_TIKTOKEN = False
if RUN_TIKTOKEN:
    import importlib.metadata
    import tiktoken

    print("tiktoken version:", importlib.metadata.version("tiktoken"))
    for encoding_name in ["cl100k_base", "o200k_base"]:
        encoding = tiktoken.get_encoding(encoding_name)
        print("\nEncoding:", encoding_name, "vocabulary:", encoding.n_vocab)
        for text in ["hello hello", "你好，世界！", "2026-09-09", "🙂", "e\u0301"]:
            ids = encoding.encode_ordinary(text)
            pieces = [encoding.decode_single_token_bytes(index) for index in ids]
            assert encoding.decode(ids) == text
            print(repr(text), "tokens:", len(ids), "IDs:", ids, "pieces:", pieces)

## References

- [Fudan Spring Lecture 01](https://baojian.github.io/llm-26/slides/lecture-01-slides/):
  language ambiguity, camera reviews, translation, and the original Ollama demonstrations.
  Task 4 in the application gallery uses the original Big Data image;
  see [media provenance](assets/README.md).
- [Stanford CS336 Spring 2026, Lecture 1](https://cs336.stanford.edu/lectures/?trace=lecture_01):
  tokenizer interfaces, byte baselines, BPE training/encoding, and resource tradeoffs.
- [Jurafsky and Martin, Words and Tokens](https://web.stanford.edu/~jurafsky/slp3/2.pdf):
  Sections 2.3–2.4 (August 2026 draft).
- [Sennrich, Haddow, and Birch (2016)](https://aclanthology.org/P16-1162/), Section 3.2;
  [Kudo and Richardson (2018)](https://aclanthology.org/P18-5009/) for further reading.
  PDF copies are in the course repository's `papers/` folder.
- [Python Unicode HOWTO](https://docs.python.org/3/howto/unicode.html).
- Ollama: [generate](https://docs.ollama.com/api/generate),
  [model list](https://docs.ollama.com/api/tags),
  [thinking](https://docs.ollama.com/capabilities/thinking),
  [vision](https://docs.ollama.com/capabilities/vision),
  [chat](https://docs.ollama.com/api/chat), and [usage metrics](https://docs.ollama.com/api/usage).
- [tiktoken](https://github.com/openai/tiktoken): named encodings and special-token handling.

API examples checked against the linked documentation on September 8, 2026.